### Update locally stored NIPA tables

In [1]:
import sys
sys.path.append('../src')

import uschartbook.config

from uschartbook.config import *
from uschartbook.utils import *

In [2]:
# Request NIPA tables data from BEA API
table_list = ['T20304', 'T70100', 'T10101', 'T10502', 
              'T11706', 'T11000', 'T20100', 'T10105',
              'T10106', 'T50100', 'T20302', 'T40202', 
              'T40100', 'T11400', 'T11200', 'T10705', 
              'T30300', 'T30200', 'T10205', 'T40205A',
              'T40205B', 'T11705', 'T30100',
              'T10103', 'T10503', 'T10505', 'T10506',
              'T10104', 'T11500']
#'T40205', 
api_results = bea_api_nipa(table_list, bea_key)

bea_to_db(api_results)

#### Update GDP estimate log

In [3]:
import csv
from pathlib import Path

log_path = Path('../gdp_estimate_log.csv')

# Get latest GDP data from database
table = retrieve_table('T10101')
gdp_line1 = [(e['TimePeriod'], e['DataValue']) for e in table['Data']
             if e['LineNumber'] == '1']
latest_quarter = gdp_line1[-1][0]

# Get the revision date from the database
conn = sqlite3.connect(db_path)
c = conn.cursor()
c.execute("SELECT MAX(date) FROM bea_nipa_raw WHERE id='T10101'")
revised_date = c.fetchone()[0]
conn.close()

# Read existing log
log_rows = []
if log_path.exists():
    with open(log_path) as f:
        reader = csv.DictReader(f)
        log_rows = list(reader)

# Check if this revision is already logged
already_logged = any(r['revised'] == revised_date for r in log_rows)

if not already_logged:
    # Determine estimate type
    prev_quarters = [r['quarter'] for r in log_rows]
    same_quarter_count = sum(1 for q in prev_quarters if q == latest_quarter)

    if same_quarter_count == 0:
        estimate = 'Advance'
    elif same_quarter_count == 1:
        estimate = 'Second'
    else:
        estimate = 'Third'

    # Append to log
    new_row = {'revised': revised_date, 'quarter': latest_quarter, 'estimate': estimate}
    write_header = not log_path.exists() or len(log_rows) == 0
    with open(log_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['revised', 'quarter', 'estimate'])
        if write_header:
            writer.writeheader()
        writer.writerow(new_row)
    print(f'{latest_quarter} — {estimate} estimate (revised {revised_date})')
else:
    estimate = next(r['estimate'] for r in log_rows if r['revised'] == revised_date)
    print(f'{latest_quarter} — {estimate} estimate (already logged)')

2025Q4 — Second estimate (revised 2026-03-13)


In [4]:
# Annual data
for table in ['T10105', 'T10104', 'T10101', 'T10502', 
              'T11000', 'T11706']:
    bea_retrieve_annual(table, bea_key)